# RFP 기반 RAG 시스템 Generation 파트 모델 및 프롬프트 실험

본 노트북은 RFP 기반 RAG 시스템에서 Generation 파트에 사용할 로컬 LLM 후보 모델과 프롬프트 기준안을 정리하기 위한 실험 노트북입니다.

Generation 파트에서는 검색된 문서 내용을 바탕으로 사용자의 질문에 답변하거나, 질문이 불완전한 경우 필요한 추가 정보를 안내해야 합니다. 또한 비교 요청, 계산 요청, 제안서 작성 요청처럼 질문 유형이 다양한 상황에서도 문서에 근거한 안정적인 응답을 생성해야 합니다.

본 실험의 목적은 다음과 같습니다.

- RFP 문서 기반 QA에 적합한 로컬 LLM 후보 비교
- 정보부족 판단 및 질문 재작성 가능 여부 확인
- 사용자에게 보여줄 수 있는 자연스러운 정보부족 안내 응답 설계
- 모델별 응답 속도와 출력 안정성 비교
- 후처리 함수의 필요성과 한계 확인
- RAG 연동 전 Generation 파트의 입력·출력 기준안 정리

본 노트북은 실험 결과를 다시 실행할 수 있도록 구성하며, 출력 결과를 확인한 뒤 해석과 최종 판단을 작성합니다. 따라서 실행 결과가 이전에 관찰한 결과와 달라질 경우, 최종 해석도 재실행 결과를 기준으로 수정합니다.

> 본 노트북은 최종 모델 선정 전 단계의 Generation 단독 실험을 정리한 노트북입니다.  
> 최종 상위 모델 비교 및 최종 모델 선정 과정은 `upper_model_comparison.ipynb`에서 다룹니다.

## 실험 진행 원칙

본 실험에서는 모델별 비교가 가능하도록 다음 원칙을 적용합니다.

- 동일한 테스트 질문을 여러 모델에 적용합니다.
- 모델 비교 시 프롬프트와 테스트 질문은 고정하고, 모델만 변경합니다.
- 프롬프트 비교 시 모델과 테스트 질문은 고정하고, 프롬프트 구조만 변경합니다.
- `temperature`는 `0.0`으로 고정하여 응답의 무작위성을 줄입니다.
- `success=True`는 모델 호출 성공을 의미하며, 응답 품질이 통과했다는 의미는 아닙니다.
- 응답 품질은 출력 내용을 직접 확인하고 수동 평가로 판단합니다.
- 최종 프롬프트는 실제 RAG 검색 결과 context 형식이 확정된 이후 다시 조정합니다.

In [ ]:
import time
import re
import subprocess
import requests
import pandas as pd

In [ ]:
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_GENERATE_URL = f"{OLLAMA_BASE_URL}/api/generate"
OLLAMA_TAGS_URL = f"{OLLAMA_BASE_URL}/api/tags"

def check_ollama_server(timeout=5):
    """
    Ollama 서버가 정상적으로 실행 중인지 확인합니다.
    """
    try:
        response = requests.get(OLLAMA_TAGS_URL, timeout=timeout)
        
        if response.status_code == 200:
            print("Ollama 서버 연결: 정상")
            return response.json()
        
        print(f"Ollama 서버 응답 이상: status_code={response.status_code}")
        print(response.text[:300])
        return None
    
    except Exception as e:
        print("Ollama 서버 연결 실패")
        print(e)
        return None

ollama_server_info = check_ollama_server()

In [ ]:
def show_ollama_models():
    """
    현재 로컬 Ollama에 설치된 모델 목록을 확인합니다.
    """
    try:
        result = subprocess.run(
            ["ollama", "list"],
            capture_output=True,
            text=True,
            encoding="utf-8"
        )
        
        print(result.stdout)
        return result.stdout
    
    except Exception as e:
        print("ollama list 실행 실패")
        print(e)
        return ""

ollama_list_text = show_ollama_models()

In [ ]:
target_models = [
    "gemma3:4b",
    "qwen2.5:3b",
    "llama3.2:3b",
    "exaone3.5:2.4b",
    "phi3.5"
]

model_check_results = []

for model in target_models:
    installed = model in ollama_list_text
    
    model_check_results.append({
        "model": model,
        "installed": installed
    })

model_check_df = pd.DataFrame(model_check_results)
model_check_df

In [ ]:
def ask_ollama(prompt, model, temperature=0.0, timeout=120):
    """
    Ollama /api/generate를 사용해 모델 응답을 생성합니다.
    
    success=True는 API 호출 성공을 의미하며,
    응답 품질이 평가 기준을 통과했다는 의미는 아닙니다.
    """
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": temperature
        }
    }
    
    start_time = time.time()
    
    try:
        response = requests.post(
            OLLAMA_GENERATE_URL,
            json=payload,
            timeout=timeout
        )
        
        elapsed = time.time() - start_time
        
        if response.status_code != 200:
            return {
                "success": False,
                "model": model,
                "answer": "",
                "elapsed": elapsed,
                "error": f"status_code={response.status_code}, text={response.text[:300]}"
            }
        
        data = response.json()
        
        return {
            "success": True,
            "model": model,
            "answer": data.get("response", "").strip(),
            "elapsed": elapsed,
            "error": ""
        }
    
    except Exception as e:
        elapsed = time.time() - start_time
        
        return {
            "success": False,
            "model": model,
            "answer": "",
            "elapsed": elapsed,
            "error": str(e)
        }

1. 환경 설정 및 모델 연결 확인

In [ ]:
smoke_test_prompt = """
다음 질문에 한국어 존댓말로 한 문장만 답변하세요.

질문: RFP 문서 기반 QA 실험에서 금액 단위 보존이 중요한 이유는 무엇인가요?
"""

smoke_test_results = []

for model in target_models:
    print(f"테스트 중: {model}")
    
    result = ask_ollama(
        prompt=smoke_test_prompt,
        model=model,
        temperature=0.0,
        timeout=120
    )
    
    smoke_test_results.append({
        "model": model,
        "success": result["success"],
        "elapsed": round(result["elapsed"], 3),
        "answer": result["answer"],
        "error": result["error"]
    })

smoke_test_df = pd.DataFrame(smoke_test_results)
smoke_test_df

2. 테스트셋 설계

In [ ]:
qa_prompt = """
너는 RFP 문서를 분석하는 AI입니다.
아래 문서 내용만 근거로 사용자의 질문에 답변하세요.

규칙:
1. 문서에 있는 내용만 사용하세요.
2. 문서에 없는 내용은 추측하지 마세요.
3. 금액, 예산, 비율, 점수, 기간 등 숫자는 원문 표기를 그대로 유지하세요.
4. 억/조 단위로 임의 변환하지 마세요.
5. 목록형 요구사항은 항목을 누락하지 말고 모두 포함하세요.
6. 가격점수 계산은 필요한 산식과 입력값이 모두 있을 때만 계산하세요.
7. 계산에 필요한 정보가 부족하면 계산하지 말고 부족한 정보를 말하세요.
8. 반드시 한국어로 답변하세요.
9. 외국어를 섞지 마세요.

[문서 내용]
{context}

[질문]
{question}

[답변]
"""

In [ ]:
clarification_prompt = """
너는 RFP 기반 질의응답 시스템의 질문 완전성 검사기입니다.
사용자의 질문이 현재 정보만으로 답변 가능한지 판단하세요.

규칙:
1. 질문이 너무 모호하면 추가 확인 필요로 판단하세요.
2. 입찰 적합도, 수행 가능성, 제안서 작성, 인력 계획 등은 회사 정보가 필요할 수 있습니다.
3. 필요한 정보가 부족하면 실제로 필요한 정보를 구체적으로 나열하세요.
4. 추가 확인이 필요한데 '없습니다'라고 쓰지 마세요.
5. 문서에 없는 정보를 추측하지 마세요.
6. 반드시 한국어로만 답변하세요.
7. 외국어를 섞지 마세요.

출력 형식:
판단: 답변 가능 / 추가 확인 필요
추가로 확인할 정보:
- 필요한 정보 1
- 필요한 정보 2

[사용자 질문]
{question}
"""

In [ ]:
rewrite_prompt = """
너는 사용자의 이전 질문과 추가 답변을 바탕으로 RFP 검색용 질문을 재작성하는 AI입니다.

규칙:
1. 이전 질문과 추가 답변을 합쳐서 더 명확한 질문으로 재작성하세요.
2. 추가 답변만으로도 정보가 부족하면 재작성 가능으로 판단하지 마세요.
3. 비교 질문은 비교 대상이 명확해야 재작성 가능합니다.
4. 입찰 적합도 판단은 회사 정보가 없으면 추가 확인 필요로 판단하세요.
5. 필요한 정보가 부족하면 실제로 필요한 정보를 구체적으로 나열하세요.
6. 반드시 한국어로만 답변하세요.
7. 외국어를 섞지 마세요.

출력 형식:
판단: 재작성 가능 / 추가 확인 필요
재작성 질문:
추가로 확인할 정보:

[이전 질문]
{previous_question}

[추가 답변]
{additional_answer}
"""

In [ ]:
qa_test_questions = [
    {
        "case_id": "QA-01",
        "category": "금액 단위 보존",
        "context": """
사업예산: 11,270,000,000원
본 사업의 예산은 부가가치세를 포함한 금액입니다.
""",
        "question": "이 사업의 예산은 얼마인가요?",
        "check_points": ["11,270,000,000원", "억/조 임의 환산 금지"]
    },
    {
        "case_id": "QA-02",
        "category": "항목 누락 방지",
        "context": """
구축 대상 시스템은 다음과 같습니다.
1. 포털 시스템
2. 학사 시스템
3. 수강신청 시스템
4. 전자결재 시스템
5. 규정관리 시스템
6. 검색 시스템
7. 챗봇 시스템
8. 모바일 시스템
9. 통계 시스템
""",
        "question": "구축 대상 시스템 목록을 모두 알려주세요.",
        "check_points": ["9개 항목 모두 포함", "항목 누락 없음"]
    },
    {
        "case_id": "QA-03",
        "category": "가격점수 가드레일",
        "context": """
가격평가는 입찰가격 평점산식에 따라 산정합니다.
다만 현재 문서에는 예정가격, 입찰가격, 평점산식의 구체적인 입력값이 모두 제시되어 있지 않습니다.
""",
        "question": "가격점수를 계산해주세요.",
        "check_points": ["계산하지 않음", "필요 정보 부족 안내"]
    }
]

qa_test_df = pd.DataFrame(qa_test_questions)
qa_test_df

In [ ]:
clarification_test_questions = [
    {
        "case_id": "CL-01",
        "category": "입찰 적합도 정보부족",
        "question": "우리 회사가 이 입찰에 적합한지 봐줘.",
        "check_points": ["회사 정보 필요", "수행 실적", "보유 인력", "기술 역량"]
    },
    {
        "case_id": "CL-02",
        "category": "제안서 작성 정보부족",
        "question": "제안서에 쓸 만하게 정리해줘.",
        "check_points": ["사업 범위", "회사 정보", "수행 전략", "실적"]
    },
    {
        "case_id": "CL-03",
        "category": "모호한 요약 요청",
        "question": "이거 핵심만 뽑아줘.",
        "check_points": ["대상 문서 또는 내용 필요"]
    },
    {
        "case_id": "CL-04",
        "category": "가격점수 정보부족",
        "question": "가격점수 계산해줘.",
        "check_points": ["입찰가격", "예정가격", "산식"]
    }
]

clarification_test_df = pd.DataFrame(clarification_test_questions)
clarification_test_df

In [ ]:
rewrite_test_cases = [
    {
        "case_id": "RW-01",
        "category": "비교 질문 재작성 가능",
        "previous_question": "이 사업과 비교해주세요.",
        "additional_answer": "경희대 산학협력단 정보시스템 운영 용역과 비교해주세요.",
        "check_points": ["재작성 가능", "비교 대상 포함"]
    },
    {
        "case_id": "RW-02",
        "category": "입찰 적합도 추가 정보 부족",
        "previous_question": "우리 회사가 이 입찰에 적합한지 봐줘.",
        "additional_answer": "그냥 알아서 해줘.",
        "check_points": ["추가 확인 필요", "회사 정보 필요", "없습니다 금지"]
    },
    {
        "case_id": "RW-03",
        "category": "수행 전략 재작성 가능",
        "previous_question": "수행 전략 작성해줘.",
        "additional_answer": "차세대 포털·학사 정보시스템 구축사업 기준으로 작성해줘.",
        "check_points": ["재작성 가능", "사업명 포함"]
    },
    {
        "case_id": "RW-04",
        "category": "비교 대상 부족",
        "previous_question": "이 사업이랑 다른 사업 비교해줘.",
        "additional_answer": "그냥 비슷한 걸로 해줘.",
        "check_points": ["추가 확인 필요", "비교 대상 필요"]
    }
]

rewrite_test_df = pd.DataFrame(rewrite_test_cases)
rewrite_test_df

3. 기본 프롬프트 모델별 실행

In [ ]:
qa_results = []

for model in target_models:
    print(f"QA 테스트 실행 중: {model}")
    
    for item in qa_test_questions:
        prompt = qa_prompt.format(
            context=item["context"],
            question=item["question"]
        )
        
        result = ask_ollama(
            prompt=prompt,
            model=model,
            temperature=0.0,
            timeout=120
        )
        
        qa_results.append({
            "model": model,
            "test_type": "QA",
            "case_id": item["case_id"],
            "category": item["category"],
            "question": item["question"],
            "check_points": ", ".join(item["check_points"]),
            "success": result["success"],
            "elapsed": round(result["elapsed"], 3),
            "answer": result["answer"],
            "error": result["error"]
        })

qa_results_df = pd.DataFrame(qa_results)
qa_results_df

In [ ]:
clarification_results = []

for model in target_models:
    print(f"정보부족 판단 테스트 실행 중: {model}")
    
    for item in clarification_test_questions:
        prompt = clarification_prompt.format(
            question=item["question"]
        )
        
        result = ask_ollama(
            prompt=prompt,
            model=model,
            temperature=0.0,
            timeout=120
        )
        
        clarification_results.append({
            "model": model,
            "test_type": "Clarification",
            "case_id": item["case_id"],
            "category": item["category"],
            "question": item["question"],
            "check_points": ", ".join(item["check_points"]),
            "success": result["success"],
            "elapsed": round(result["elapsed"], 3),
            "answer": result["answer"],
            "error": result["error"]
        })

clarification_results_df = pd.DataFrame(clarification_results)
clarification_results_df

In [ ]:
rewrite_results = []

for model in target_models:
    print(f"질문 재작성 테스트 실행 중: {model}")
    
    for item in rewrite_test_cases:
        prompt = rewrite_prompt.format(
            previous_question=item["previous_question"],
            additional_answer=item["additional_answer"]
        )
        
        result = ask_ollama(
            prompt=prompt,
            model=model,
            temperature=0.0,
            timeout=120
        )
        
        rewrite_results.append({
            "model": model,
            "test_type": "Rewrite",
            "case_id": item["case_id"],
            "category": item["category"],
            "previous_question": item["previous_question"],
            "additional_answer": item["additional_answer"],
            "check_points": ", ".join(item["check_points"]),
            "success": result["success"],
            "elapsed": round(result["elapsed"], 3),
            "answer": result["answer"],
            "error": result["error"]
        })

rewrite_results_df = pd.DataFrame(rewrite_results)
rewrite_results_df

In [ ]:
all_test_results_df = pd.concat(
    [
        qa_results_df,
        clarification_results_df,
        rewrite_results_df
    ],
    ignore_index=True,
    sort=False
)

all_test_results_df[
    ["model", "test_type", "case_id", "category", "success", "elapsed", "answer"]
]

In [ ]:
model_time_summary_df = all_test_results_df.groupby("model")["elapsed"].agg(
    count="count",
    mean="mean",
    min="min",
    max="max"
).reset_index()

model_time_summary_df = model_time_summary_df.sort_values("mean")
model_time_summary_df

In [ ]:
test_type_time_summary_df = all_test_results_df.groupby(
    ["model", "test_type"]
)["elapsed"].agg(
    count="count",
    mean="mean",
    min="min",
    max="max"
).reset_index()

test_type_time_summary_df.sort_values(["test_type", "mean"])

In [ ]:
call_success_summary_df = all_test_results_df.groupby(
    ["model", "success"]
).size().reset_index(name="count")

call_success_summary_df

In [ ]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

all_test_results_df[
    ["model", "test_type", "case_id", "category", "success", "elapsed", "answer"]
]

In [ ]:
review_cases = all_test_results_df[
    all_test_results_df["case_id"].isin([
        "QA-02", 
        "CL-01", "CL-02", "CL-04",
        "RW-02", "RW-04"
    ])
][
    ["model", "test_type", "case_id", "category", "answer"]
]

review_cases

In [ ]:
for _, row in review_cases.iterrows():
    print("=" * 80)
    print(f"model: {row['model']}")
    print(f"test_type: {row['test_type']}")
    print(f"case_id: {row['case_id']}")
    print(f"category: {row['category']}")
    print("-" * 80)
    print(row["answer"])
    print()

4. 수동 평가

In [ ]:
manual_eval_df = all_test_results_df[
    ["model", "test_type", "case_id", "category", "answer"]
].copy()

manual_eval_df["manual_result"] = ""
manual_eval_df["memo"] = ""

manual_eval_df

In [ ]:
manual_eval_records = [
    # gemma3:4b
    {
        "model": "gemma3:4b",
        "test_type": "QA",
        "case_id": "QA-01",
        "manual_result": "통과",
        "memo": "금액을 11,270,000,000원으로 원문 그대로 유지함"
    },
    {
        "model": "gemma3:4b",
        "test_type": "QA",
        "case_id": "QA-02",
        "manual_result": "통과",
        "memo": "구축 대상 시스템 9개 항목을 모두 출력함"
    },
    {
        "model": "gemma3:4b",
        "test_type": "QA",
        "case_id": "QA-03",
        "manual_result": "통과",
        "memo": "예정가격, 입찰가격, 평점산식 입력값 부족으로 계산하지 않음"
    },
    {
        "model": "gemma3:4b",
        "test_type": "Clarification",
        "case_id": "CL-01",
        "manual_result": "부분 통과",
        "memo": "추가 확인 필요 판단은 적절하나 수행 실적, 보유 인력, 기술 역량이 직접적으로 충분히 제시되지는 않음"
    },
    {
        "model": "gemma3:4b",
        "test_type": "Clarification",
        "case_id": "CL-02",
        "manual_result": "부분 통과",
        "memo": "추가 확인 필요 판단은 적절하나 회사 정보, 수행 전략, 실적 안내가 부족함"
    },
    {
        "model": "gemma3:4b",
        "test_type": "Clarification",
        "case_id": "CL-03",
        "manual_result": "통과",
        "memo": "모호한 요약 요청에 대해 대상 내용과 핵심 기준 확인이 필요하다고 안내함"
    },
    {
        "model": "gemma3:4b",
        "test_type": "Clarification",
        "case_id": "CL-04",
        "manual_result": "부분 통과",
        "memo": "추가 확인 필요 판단은 맞지만 입찰가격, 예정가격, 산식을 직접적으로 모두 짚지는 못함"
    },
    {
        "model": "gemma3:4b",
        "test_type": "Rewrite",
        "case_id": "RW-01",
        "manual_result": "통과",
        "memo": "비교 대상이 명확한 상황에서 재작성 가능으로 판단하고 비교 질문을 생성함"
    },
    {
        "model": "gemma3:4b",
        "test_type": "Rewrite",
        "case_id": "RW-02",
        "manual_result": "부분 통과",
        "memo": "회사 정보 부족을 안내했지만 판단이 재작성 가능 / 추가 확인 필요로 혼재됨"
    },
    {
        "model": "gemma3:4b",
        "test_type": "Rewrite",
        "case_id": "RW-03",
        "manual_result": "통과",
        "memo": "사업명을 반영하여 수행 전략 질문으로 재작성함"
    },
    {
        "model": "gemma3:4b",
        "test_type": "Rewrite",
        "case_id": "RW-04",
        "manual_result": "실패",
        "memo": "비교 대상이 부족한데 재작성 가능으로 오판함"
    },

    # qwen2.5:3b
    {
        "model": "qwen2.5:3b",
        "test_type": "QA",
        "case_id": "QA-01",
        "manual_result": "통과",
        "memo": "금액을 11,270,000,000원으로 원문 그대로 유지함"
    },
    {
        "model": "qwen2.5:3b",
        "test_type": "QA",
        "case_id": "QA-02",
        "manual_result": "통과",
        "memo": "구축 대상 시스템 9개 항목을 모두 출력함"
    },
    {
        "model": "qwen2.5:3b",
        "test_type": "QA",
        "case_id": "QA-03",
        "manual_result": "통과",
        "memo": "예정가격, 입찰가격, 평점산식 입력값 부족으로 계산하지 않음"
    },
    {
        "model": "qwen2.5:3b",
        "test_type": "Clarification",
        "case_id": "CL-01",
        "manual_result": "부분 통과",
        "memo": "추가 확인 필요 판단은 맞지만 수행 실적, 보유 인력 등 구체성이 부족함"
    },
    {
        "model": "qwen2.5:3b",
        "test_type": "Clarification",
        "case_id": "CL-02",
        "manual_result": "부분 통과",
        "memo": "추가 확인 필요 판단은 맞지만 회사 정보, 수행 전략, 실적 안내가 부족함"
    },
    {
        "model": "qwen2.5:3b",
        "test_type": "Clarification",
        "case_id": "CL-03",
        "manual_result": "부분 통과",
        "memo": "추가 확인 필요 판단은 맞지만 대상 문서나 내용 필요성을 충분히 구체화하지 못함"
    },
    {
        "model": "qwen2.5:3b",
        "test_type": "Clarification",
        "case_id": "CL-04",
        "manual_result": "부분 통과",
        "memo": "추가 확인 필요 판단은 맞지만 입찰가격, 예정가격, 산식이라는 핵심 용어가 직접적으로 부족함"
    },
    {
        "model": "qwen2.5:3b",
        "test_type": "Rewrite",
        "case_id": "RW-01",
        "manual_result": "부분 통과",
        "memo": "비교 대상은 반영했지만 판단이 재작성 가능 / 추가 확인 필요로 혼재됨"
    },
    {
        "model": "qwen2.5:3b",
        "test_type": "Rewrite",
        "case_id": "RW-02",
        "manual_result": "부분 통과",
        "memo": "재작성 불가능하다는 방향은 맞지만 출력 형식이 지정 형식과 다름"
    },
    {
        "model": "qwen2.5:3b",
        "test_type": "Rewrite",
        "case_id": "RW-03",
        "manual_result": "통과",
        "memo": "사업명을 반영하여 수행 전략 관련 질문으로 재작성함"
    },
    {
        "model": "qwen2.5:3b",
        "test_type": "Rewrite",
        "case_id": "RW-04",
        "manual_result": "부분 통과",
        "memo": "추가 확인 필요 판단은 맞지만 재작성 질문을 함께 출력하여 형식이 혼재됨"
    },

    # llama3.2:3b
    {
        "model": "llama3.2:3b",
        "test_type": "QA",
        "case_id": "QA-01",
        "manual_result": "통과",
        "memo": "금액을 11,270,000,000원으로 원문 그대로 유지함"
    },
    {
        "model": "llama3.2:3b",
        "test_type": "QA",
        "case_id": "QA-02",
        "manual_result": "통과",
        "memo": "구축 대상 시스템 9개 항목을 모두 출력함"
    },
    {
        "model": "llama3.2:3b",
        "test_type": "QA",
        "case_id": "QA-03",
        "manual_result": "실패",
        "memo": "가격점수를 계산하지 않은 방향은 맞지만 중국어 표현이 혼입됨"
    },
    {
        "model": "llama3.2:3b",
        "test_type": "Clarification",
        "case_id": "CL-01",
        "manual_result": "실패",
        "memo": "추가 확인 필요 상황을 답변 가능으로 오판함"
    },
    {
        "model": "llama3.2:3b",
        "test_type": "Clarification",
        "case_id": "CL-02",
        "manual_result": "실패",
        "memo": "추가 확인 필요 상황을 답변 가능으로 오판함"
    },
    {
        "model": "llama3.2:3b",
        "test_type": "Clarification",
        "case_id": "CL-03",
        "manual_result": "실패",
        "memo": "모호한 요약 요청을 답변 가능으로 판단하고 추가 정보 없음으로 출력함"
    },
    {
        "model": "llama3.2:3b",
        "test_type": "Clarification",
        "case_id": "CL-04",
        "manual_result": "실패",
        "memo": "가격점수 계산 요청을 답변 가능으로 판단하고 추가 정보 없음으로 출력함"
    },
    {
        "model": "llama3.2:3b",
        "test_type": "Rewrite",
        "case_id": "RW-01",
        "manual_result": "부분 통과",
        "memo": "재작성 가능 판단은 맞지만 비교 대상 해석이 부정확함"
    },
    {
        "model": "llama3.2:3b",
        "test_type": "Rewrite",
        "case_id": "RW-02",
        "manual_result": "실패",
        "memo": "회사 정보 부족 상황을 재작성 가능으로 오판하고 영어 표현이 혼입됨"
    },
    {
        "model": "llama3.2:3b",
        "test_type": "Rewrite",
        "case_id": "RW-03",
        "manual_result": "부분 통과",
        "memo": "사업명은 반영했지만 scope, timeline, deadlines 등 영어 표현이 혼입됨"
    },
    {
        "model": "llama3.2:3b",
        "test_type": "Rewrite",
        "case_id": "RW-04",
        "manual_result": "실패",
        "memo": "비교 대상 부족 상황을 재작성 가능으로 오판함"
    },

    # exaone3.5:2.4b
    {
        "model": "exaone3.5:2.4b",
        "test_type": "QA",
        "case_id": "QA-01",
        "manual_result": "통과",
        "memo": "금액을 11,270,000,000원으로 원문 그대로 유지함"
    },
    {
        "model": "exaone3.5:2.4b",
        "test_type": "QA",
        "case_id": "QA-02",
        "manual_result": "통과",
        "memo": "구축 대상 시스템 9개 항목을 모두 출력함"
    },
    {
        "model": "exaone3.5:2.4b",
        "test_type": "QA",
        "case_id": "QA-03",
        "manual_result": "부분 통과",
        "memo": "계산하지 않은 점은 적절하나 부족 정보에서 입찰가격이 명확히 빠짐"
    },
    {
        "model": "exaone3.5:2.4b",
        "test_type": "Clarification",
        "case_id": "CL-01",
        "manual_result": "부분 통과",
        "memo": "추가 확인 필요 판단은 맞지만 회사 수행 실적, 보유 인력, 기술 역량보다 일반 입찰 정보 중심임"
    },
    {
        "model": "exaone3.5:2.4b",
        "test_type": "Clarification",
        "case_id": "CL-02",
        "manual_result": "부분 통과",
        "memo": "추가 확인 필요 판단은 맞지만 회사 정보와 실적보다 프로젝트 정보 중심임"
    },
    {
        "model": "exaone3.5:2.4b",
        "test_type": "Clarification",
        "case_id": "CL-03",
        "manual_result": "통과",
        "memo": "모호한 요약 요청에 대해 구체적인 RFP 요구사항과 응답 범위를 확인함"
    },
    {
        "model": "exaone3.5:2.4b",
        "test_type": "Clarification",
        "case_id": "CL-04",
        "manual_result": "부분 통과",
        "memo": "추가 확인 필요 판단은 맞지만 가격점수 계산에 필요한 입찰가격, 예정가격, 산식을 직접적으로 짚지 못함"
    },
    {
        "model": "exaone3.5:2.4b",
        "test_type": "Rewrite",
        "case_id": "RW-01",
        "manual_result": "부분 통과",
        "memo": "비교 대상은 반영했지만 RFP 영어 설명이 포함되고 추가 정보가 과도함"
    },
    {
        "model": "exaone3.5:2.4b",
        "test_type": "Rewrite",
        "case_id": "RW-02",
        "manual_result": "부분 통과",
        "memo": "추가 확인 필요 판단은 맞지만 재작성 질문을 함께 생성하고 영어 표현이 포함됨"
    },
    {
        "model": "exaone3.5:2.4b",
        "test_type": "Rewrite",
        "case_id": "RW-03",
        "manual_result": "통과",
        "memo": "사업명을 반영하여 수행 전략 질문으로 재작성함"
    },
    {
        "model": "exaone3.5:2.4b",
        "test_type": "Rewrite",
        "case_id": "RW-04",
        "manual_result": "실패",
        "memo": "비교 대상 부족 상황을 재작성 가능으로 오판함"
    },

    # phi3.5
    {
        "model": "phi3.5",
        "test_type": "QA",
        "case_id": "QA-01",
        "manual_result": "통과",
        "memo": "금액 자체는 원문 그대로 유지함"
    },
    {
        "model": "phi3.5",
        "test_type": "QA",
        "case_id": "QA-02",
        "manual_result": "실패",
        "memo": "챗봇 시스템을 챕터블러 시스템으로 잘못 출력함"
    },
    {
        "model": "phi3.5",
        "test_type": "QA",
        "case_id": "QA-03",
        "manual_result": "실패",
        "memo": "깨진 표현과 의미 불명확한 문장이 포함됨"
    },
    {
        "model": "phi3.5",
        "test_type": "Clarification",
        "case_id": "CL-01",
        "manual_result": "부분 통과",
        "memo": "추가 확인 필요 판단은 맞지만 표현이 부자연스럽고 요구 정보가 정돈되지 않음"
    },
    {
        "model": "phi3.5",
        "test_type": "Clarification",
        "case_id": "CL-02",
        "manual_result": "부분 통과",
        "memo": "추가 확인 필요 판단은 맞지만 회사 정보와 수행 실적 안내가 부족함"
    },
    {
        "model": "phi3.5",
        "test_type": "Clarification",
        "case_id": "CL-03",
        "manual_result": "실패",
        "memo": "RFP를 공개선서로 잘못 표현하는 등 응답 품질 문제가 있음"
    },
    {
        "model": "phi3.5",
        "test_type": "Clarification",
        "case_id": "CL-04",
        "manual_result": "실패",
        "memo": "판단이 혼재되고 내부 라벨을 노출하며 질문과 맞지 않는 정보를 요구함"
    },
    {
        "model": "phi3.5",
        "test_type": "Rewrite",
        "case_id": "RW-01",
        "manual_result": "부분 통과",
        "memo": "비교 대상은 일부 반영했지만 질문 문장이 부자연스러움"
    },
    {
        "model": "phi3.5",
        "test_type": "Rewrite",
        "case_id": "RW-02",
        "manual_result": "실패",
        "memo": "회사 정보 부족 상황을 재작성 가능으로 오판하고 문장이 부자연스러움"
    },
    {
        "model": "phi3.5",
        "test_type": "Rewrite",
        "case_id": "RW-03",
        "manual_result": "부분 통과",
        "memo": "사업명은 반영했지만 재작성 질문이 단순하고 구체성이 부족함"
    },
    {
        "model": "phi3.5",
        "test_type": "Rewrite",
        "case_id": "RW-04",
        "manual_result": "실패",
        "memo": "비교 대상 부족 상황을 재작성 가능으로 오판하고 문장이 부자연스러움"
    }
]

manual_eval_result_df = pd.DataFrame(manual_eval_records)
manual_eval_result_df

In [ ]:
manual_eval_summary_df = manual_eval_result_df.groupby(
    ["model", "manual_result"]
).size().reset_index(name="count")

manual_eval_summary_df

In [ ]:
manual_eval_by_type_df = manual_eval_result_df.groupby(
    ["model", "test_type", "manual_result"]
).size().reset_index(name="count")

manual_eval_by_type_df.sort_values(["model", "test_type", "manual_result"])

In [ ]:
score_map = {
    "통과": 1.0,
    "부분 통과": 0.5,
    "실패": 0.0
}

manual_eval_result_df["score"] = manual_eval_result_df["manual_result"].map(score_map)

model_score_summary_df = manual_eval_result_df.groupby("model")["score"].agg(
    test_count="count",
    total_score="sum",
    average_score="mean"
).reset_index()

model_score_summary_df = model_score_summary_df.sort_values(
    "average_score",
    ascending=False
)

model_score_summary_df

5. 사용자 응답형 안내 테스트

In [ ]:
user_facing_clarification_prompt = """
너는 RFP 분석 서비스를 사용하는 사용자에게 답변하는 AI입니다.

사용자의 질문에 답변하기 위해 정보가 부족한 경우,
내부 판단 라벨을 보여주지 말고 사용자에게 자연스럽게 추가 정보를 요청하세요.

규칙:
1. "판단:", "추가 확인 필요", "답변 가능" 같은 내부 라벨을 출력하지 마세요.
2. 사용자의 질문에 바로 답하기 어려운 이유를 간단히 설명하세요.
3. 추가로 필요한 정보를 구체적으로 안내하세요.
4. 사용자가 다음에 입력하면 좋은 예시 질문을 1개 제시하세요.
5. 문장은 한국어 존댓말로 작성하세요.
6. 외국어를 섞지 마세요.
7. 너무 길게 쓰지 말고, 4~6문장 정도로 답변하세요.
8. 문서에 없는 내용은 추측하지 마세요.

[사용자 질문]
{question}

[사용자에게 보여줄 답변]
"""

In [ ]:
user_facing_test_questions = [
    {
        "case_id": "UF-01",
        "category": "입찰 적합도 안내",
        "question": "우리 회사가 이 입찰에 적합한지 봐줘.",
        "check_points": [
            "회사 정보 필요 안내",
            "수행 실적 또는 보유 인력 안내",
            "내부 라벨 미노출",
            "자연스러운 존댓말"
        ]
    },
    {
        "case_id": "UF-02",
        "category": "제안서 작성 안내",
        "question": "제안서에 쓸 만하게 정리해줘.",
        "check_points": [
            "사업 정보 필요 안내",
            "회사 정보 필요 안내",
            "작성 방향 안내",
            "내부 라벨 미노출"
        ]
    },
    {
        "case_id": "UF-03",
        "category": "모호한 요약 요청 안내",
        "question": "이거 핵심만 뽑아줘.",
        "check_points": [
            "대상 문서 또는 내용 필요 안내",
            "요약 범위 안내",
            "예시 질문 제시",
            "자연스러운 문체"
        ]
    },
    {
        "case_id": "UF-04",
        "category": "가격점수 계산 안내",
        "question": "가격점수 계산해줘.",
        "check_points": [
            "입찰가격 필요 안내",
            "예정가격 필요 안내",
            "평점산식 필요 안내",
            "임의 계산 금지"
        ]
    }
]

user_facing_test_df = pd.DataFrame(user_facing_test_questions)
user_facing_test_df

In [ ]:
user_facing_results = []

for model in target_models:
    print(f"사용자 응답형 안내 테스트 실행 중: {model}")
    
    for item in user_facing_test_questions:
        prompt = user_facing_clarification_prompt.format(
            question=item["question"]
        )
        
        result = ask_ollama(
            prompt=prompt,
            model=model,
            temperature=0.0,
            timeout=120
        )
        
        user_facing_results.append({
            "model": model,
            "test_type": "User-Facing Clarification",
            "case_id": item["case_id"],
            "category": item["category"],
            "question": item["question"],
            "check_points": ", ".join(item["check_points"]),
            "success": result["success"],
            "elapsed": round(result["elapsed"], 3),
            "answer": result["answer"],
            "error": result["error"]
        })

user_facing_results_df = pd.DataFrame(user_facing_results)
user_facing_results_df

In [ ]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

user_facing_results_df[
    ["model", "case_id", "category", "success", "elapsed", "answer"]
]

In [ ]:
user_facing_manual_eval_records = [
    # gemma3:4b
    {
        "model": "gemma3:4b",
        "case_id": "UF-01",
        "manual_result": "부분 통과",
        "memo": "자연스러운 존댓말 응답이며 내부 라벨은 없지만 수행 실적, 보유 인력 안내가 부족함"
    },
    {
        "model": "gemma3:4b",
        "case_id": "UF-02",
        "manual_result": "부분 통과",
        "memo": "제안서 작성에 필요한 추가 정보를 안내했지만 회사 정보와 수행 전략 안내가 부족하고 영어 설명이 포함됨"
    },
    {
        "model": "gemma3:4b",
        "case_id": "UF-03",
        "manual_result": "통과",
        "memo": "대상 문서와 요약 범위가 필요하다는 점을 자연스럽게 안내함"
    },
    {
        "model": "gemma3:4b",
        "case_id": "UF-04",
        "manual_result": "부분 통과",
        "memo": "임의 계산은 하지 않았지만 입찰가격, 예정가격, 평점산식이라는 핵심 입력값을 직접적으로 모두 안내하지 못함"
    },

    # qwen2.5:3b
    {
        "model": "qwen2.5:3b",
        "case_id": "UF-01",
        "manual_result": "부분 통과",
        "memo": "필요 정보 부족을 안내했지만 응답이 짧고 수행 실적, 보유 인력 안내가 부족함"
    },
    {
        "model": "qwen2.5:3b",
        "case_id": "UF-02",
        "manual_result": "실패",
        "memo": "판정이라는 내부 라벨이 노출되고 회사 정보 안내가 부족함"
    },
    {
        "model": "qwen2.5:3b",
        "case_id": "UF-03",
        "manual_result": "부분 통과",
        "memo": "추가 정보 필요성을 안내했지만 대상 문서와 요약 범위 안내가 다소 일반적임"
    },
    {
        "model": "qwen2.5:3b",
        "case_id": "UF-04",
        "manual_result": "실패",
        "memo": "입찰가격, 예정가격, 평점산식을 직접 안내하지 못하고 현재 사용 중인 가격/비교 대상 가격으로 일반화함"
    },

    # llama3.2:3b
    {
        "model": "llama3.2:3b",
        "case_id": "UF-01",
        "manual_result": "실패",
        "memo": "requirements, details 등 영어 표현이 혼입되고 문체가 부자연스러움"
    },
    {
        "model": "llama3.2:3b",
        "case_id": "UF-02",
        "manual_result": "실패",
        "memo": "응답이 장황하고 반복적이며 사용자 응답형 문장으로 부적절함"
    },
    {
        "model": "llama3.2:3b",
        "case_id": "UF-03",
        "manual_result": "실패",
        "memo": "영어, 일본어, 힌디어로 보이는 문자가 혼입되어 사용자 응답으로 부적합함"
    },
    {
        "model": "llama3.2:3b",
        "case_id": "UF-04",
        "manual_result": "실패",
        "memo": "임의 계산 방식 예시를 제시하고 price score 등 영어 표현이 혼입됨"
    },

    # exaone3.5:2.4b
    {
        "model": "exaone3.5:2.4b",
        "case_id": "UF-01",
        "manual_result": "통과",
        "memo": "회사 역량과 경험, 자원 정보를 자연스럽게 요구함"
    },
    {
        "model": "exaone3.5:2.4b",
        "case_id": "UF-02",
        "manual_result": "부분 통과",
        "memo": "제안서 작성에 필요한 정보를 풍부하게 안내했지만 답변이 다소 길고 회사 정보 안내가 약함"
    },
    {
        "model": "exaone3.5:2.4b",
        "case_id": "UF-03",
        "manual_result": "부분 통과",
        "memo": "요약 대상과 범위를 안내했지만 RFP 영어 설명이 포함되고 답변이 다소 장황함"
    },
    {
        "model": "exaone3.5:2.4b",
        "case_id": "UF-04",
        "manual_result": "부분 통과",
        "memo": "가격점수 계산에 추가 정보가 필요하다고 안내했지만 입찰가격, 예정가격, 평점산식을 직접 짚지 못함"
    },

    # phi3.5
    {
        "model": "phi3.5",
        "case_id": "UF-01",
        "manual_result": "실패",
        "memo": "내부 라벨이 노출되고 깨진 표현 및 외국어 혼입이 있음"
    },
    {
        "model": "phi3.5",
        "case_id": "UF-02",
        "manual_result": "실패",
        "memo": "내부 라벨이 노출되고 문장이 부자연스러움"
    },
    {
        "model": "phi3.5",
        "case_id": "UF-03",
        "manual_result": "실패",
        "memo": "내부 라벨, 외국어 혼입, 부자연스러운 문장이 포함됨"
    },
    {
        "model": "phi3.5",
        "case_id": "UF-04",
        "manual_result": "실패",
        "memo": "응답이 과도하게 반복되고 일본어/영어/깨진 표현이 혼입되어 사용 불가"
    }
]

user_facing_manual_eval_result_df = pd.DataFrame(user_facing_manual_eval_records)
user_facing_manual_eval_result_df

In [ ]:
user_facing_eval_summary_df = user_facing_manual_eval_result_df.groupby(
    ["model", "manual_result"]
).size().reset_index(name="count")

user_facing_eval_summary_df

In [ ]:
user_facing_score_map = {
    "통과": 1.0,
    "부분 통과": 0.5,
    "실패": 0.0
}

user_facing_manual_eval_result_df["score"] = user_facing_manual_eval_result_df["manual_result"].map(user_facing_score_map)

user_facing_score_summary_df = user_facing_manual_eval_result_df.groupby("model")["score"].agg(
    test_count="count",
    total_score="sum",
    average_score="mean"
).reset_index()

user_facing_score_summary_df = user_facing_score_summary_df.sort_values(
    "average_score",
    ascending=False
)

user_facing_score_summary_df

In [ ]:
final_model_comparison_df = pd.DataFrame([
    {
        "model": "gemma3:4b",
        "role": "품질 중심 주력 후보",
        "strength": "문서 기반 QA 안정성, 자연스러운 한국어 존댓말 응답, 정보부족 판단 방향성",
        "limitation": "질문 재작성에서 정보부족 상황을 재작성 가능으로 오판한 사례가 있음. 일부 안내 항목의 구체성 보완 필요",
        "final_decision": "주력 후보"
    },
    {
        "model": "qwen2.5:3b",
        "role": "속도 중심 대안 후보",
        "strength": "빠른 응답 속도, 문서 기반 QA 안정성, 전체 수동 평가에서 실패 케이스 없음",
        "limitation": "사용자 응답형 안내에서 내부 라벨 노출과 정보 요구 부정확성 확인. 출력 형식 제어 필요",
        "final_decision": "대안 후보"
    },
    {
        "model": "exaone3.5:2.4b",
        "role": "보조 후보 / 추가 검증 후보",
        "strength": "사용자 응답형 안내문 생성 능력, QA 및 정보부족 판단의 기본 가능성",
        "limitation": "답변이 장황하고 영어 설명이 포함됨. 질문 재작성에서 정보부족 상황 오판 사례 있음",
        "final_decision": "추가 검증 필요"
    },
    {
        "model": "llama3.2:3b",
        "role": "제외 후보",
        "strength": "일부 단순 QA에서는 금액 보존 및 항목 나열 가능",
        "limitation": "정보부족 판단 실패, 외국어 혼입, 사용자 응답형 안내 품질 문제",
        "final_decision": "제외"
    },
    {
        "model": "phi3.5",
        "role": "제외 후보",
        "strength": "금액 보존과 일부 단순 응답 가능",
        "limitation": "항목명 오류, 깨진 표현, 내부 라벨 노출, 반복 출력, 외국어 혼입",
        "final_decision": "제외"
    }
])

final_model_comparison_df

In [ ]:
base_score_df = model_score_summary_df.rename(
    columns={
        "average_score": "base_average_score",
        "total_score": "base_total_score",
        "test_count": "base_test_count"
    }
)

user_facing_score_df = user_facing_score_summary_df.rename(
    columns={
        "average_score": "user_facing_average_score",
        "total_score": "user_facing_total_score",
        "test_count": "user_facing_test_count"
    }
)

combined_score_df = pd.merge(
    base_score_df,
    user_facing_score_df,
    on="model",
    how="left"
)

combined_score_df["combined_average_score"] = (
    combined_score_df["base_average_score"] + combined_score_df["user_facing_average_score"]
) / 2

combined_score_df = combined_score_df.sort_values(
    "combined_average_score",
    ascending=False
)

combined_score_df

In [ ]:
final_candidate_models = [
    "gemma3:4b",
    "qwen2.5:3b",
    "exaone3.5:2.4b"
]

final_candidate_models

6. 후처리 함수 설계

In [ ]:
def postprocess_user_response(answer):
    """
    LLM 원본 응답을 사용자에게 보여주기 전에 검수하는 함수입니다.
    
    처리 내용:
    - 빈 응답 또는 지나치게 짧은 응답 대체
    - 내부 라벨 제거
    - 외국어 문자 혼입 감지
    - 의미가 깨질 수 있는 문장 단위 자르기는 수행하지 않음
    """
    
    fallback_answer = (
        "현재 질문만으로는 정확한 답변을 드리기 어렵습니다. "
        "관련 RFP 문서나 추가 정보를 제공해주시면 확인 후 안내해 드리겠습니다."
    )
    
    if answer is None:
        return fallback_answer
    
    processed = str(answer).strip()
    
    if len(processed) < 10:
        return fallback_answer
    
    # 한자, 일본어, 키릴 문자 등 한국어 응답에 부적절한 외국어 문자 감지
    foreign_pattern = r"[\u4e00-\u9fff\u3040-\u30ff\u0400-\u04FF]"
    
    if re.search(foreign_pattern, processed):
        return fallback_answer
    
    banned_labels = [
        "판단:",
        "판정:",
        "추가 확인 필요:",
        "추가 확인 필요",
        "답변 가능",
        "재작성 질문:",
        "추가로 확인할 정보:",
        "출력:",
        "답변:",
        "문장화:",
        "[최종 답변]",
        "[응답]"
    ]
    
    for label in banned_labels:
        processed = processed.replace(label, "").strip()
    
    if len(processed) < 10:
        return fallback_answer
    
    return processed

In [ ]:
postprocess_test_samples = [
    "판단: 추가 확인 필요\n추가로 확인할 정보:\n- 입찰가격\n- 예정가격\n- 평점산식",
    "현재 질문만으로는 가격점수를 계산하기 어렵습니다. 입찰가격, 예정가격, 평점산식을 제공해 주세요.",
    "현재 문서에는 가격점수를 계산할 수 없습니다. 因此 추가 정보가 필요합니다.",
    "",
    "짧음"
]

for i, sample in enumerate(postprocess_test_samples, start=1):
    print("=" * 80)
    print(f"샘플 {i}")
    print("[원본]")
    print(sample)
    print("\n[후처리 결과]")
    print(postprocess_user_response(sample))
    print()

In [ ]:
user_facing_postprocessed_df = user_facing_results_df.copy()

user_facing_postprocessed_df["postprocessed_answer"] = user_facing_postprocessed_df["answer"].apply(
    postprocess_user_response
)

user_facing_postprocessed_df["changed"] = (
    user_facing_postprocessed_df["answer"] != user_facing_postprocessed_df["postprocessed_answer"]
)

user_facing_postprocessed_df[
    [
        "model",
        "case_id",
        "category",
        "changed",
        "answer",
        "postprocessed_answer"
    ]
]

In [ ]:
candidate_postprocessed_df = user_facing_postprocessed_df[
    user_facing_postprocessed_df["model"].isin(final_candidate_models)
].copy()

candidate_postprocessed_df[
    [
        "model",
        "case_id",
        "category",
        "changed",
        "answer",
        "postprocessed_answer"
    ]
]

In [ ]:
postprocess_change_summary_df = user_facing_postprocessed_df.groupby(
    ["model", "changed"]
).size().reset_index(name="count")

postprocess_change_summary_df

In [ ]:
candidate_postprocess_change_summary_df = candidate_postprocessed_df.groupby(
    ["model", "changed"]
).size().reset_index(name="count")

candidate_postprocess_change_summary_df

7. 개선 프롬프트 실험

In [ ]:
improved_rag_qa_prompt = """
너는 공공 RFP 문서를 분석하는 AI입니다.
반드시 아래 [검색된 문서 내용]만 근거로 사용자의 질문에 답변하세요.

핵심 규칙:
1. 문서에 있는 내용만 사용하세요.
2. 문서에 없는 내용은 추측하지 마세요.
3. 금액, 기간, 비율, 점수, 배점, 날짜는 원문 표기를 그대로 유지하세요.
4. 억/조 단위로 임의 변환하지 마세요.
5. 목록형 요구사항은 항목을 누락하지 말고 모두 포함하세요.
6. 문서에 근거가 부족하면 부족하다고 말하고, 필요한 정보를 구체적으로 안내하세요.
7. 답변은 한국어 존댓말로 작성하세요.
8. 영어, 일본어, 중국어 등 외국어를 섞지 마세요.
9. "판단:", "출력:", "추가 확인 필요" 같은 내부 라벨을 사용자에게 노출하지 마세요.

가격점수 계산 규칙:
1. 가격점수 계산은 다음 정보가 모두 있을 때만 수행하세요.
   - 입찰가격
   - 예정가격 또는 기준가격
   - 가격점수 평점산식
   - 가격 평가 배점
2. 위 정보 중 하나라도 부족하면 계산하지 마세요.
3. 계산할 수 없는 경우에는 부족한 정보를 구체적으로 안내하세요.
4. 임의의 산식이나 예시 산식을 만들어 계산하지 마세요.

[검색된 문서 내용]
{context}

[사용자 질문]
{question}

[답변]
"""

In [ ]:
improved_user_facing_clarification_prompt = """
너는 RFP 분석 서비스를 사용하는 사용자에게 답변하는 AI입니다.
사용자의 질문에 답변하기 위해 정보가 부족한 경우, 사용자에게 자연스럽게 추가 정보를 요청하세요.

공통 규칙:
1. "판단:", "판정:", "추가 확인 필요", "답변 가능" 같은 내부 라벨을 출력하지 마세요.
2. 사용자의 질문에 바로 답하기 어려운 이유를 1문장으로 설명하세요.
3. 추가로 필요한 정보를 구체적으로 안내하세요.
4. 답변은 4~6문장 이내로 작성하세요.
5. 반드시 한국어 존댓말로 작성하세요.
6. 영어 설명을 넣지 마세요.
7. 문서에 없는 내용은 추측하지 마세요.
8. 마지막에 사용자가 다음에 입력하면 좋은 예시 질문을 1개만 제시하세요.

요청 유형별 필수 확인 정보:
- 입찰 적합도 판단: 회사 주요 사업 분야, 유사 사업 수행 실적, 보유 인력, 기술 역량, 입찰 자격 충족 여부
- 제안서 작성: 대상 RFP 또는 사업명, 회사 소개, 회사 강점, 유사 사업 실적, 수행 전략, 작성할 제안서 항목
- 가격점수 계산: 입찰가격, 예정가격 또는 기준가격, 가격점수 평점산식, 가격 평가 배점
- 문서 요약: 대상 문서, 요약 범위, 중점적으로 볼 항목
- 문서 비교: 비교 대상 문서 2개 이상, 비교 기준, 보고 싶은 항목

[사용자 질문]
{question}

[사용자에게 보여줄 답변]
"""

In [ ]:
improved_rewrite_prompt = """
너는 사용자의 질문을 RFP 검색에 적합한 질문으로 재작성하는 AI입니다.

규칙:
1. 이전 질문과 추가 답변을 바탕으로 검색 가능한 질문을 만드세요.
2. 정보가 충분하지 않으면 재작성하지 마세요.
3. 비교 질문은 비교 대상이 명확할 때만 재작성하세요.
4. 입찰 적합도 판단은 회사 정보가 충분할 때만 재작성하세요.
5. "그냥 알아서 해줘", "비슷한 걸로 해줘", "아무거나"처럼 정보가 부족한 답변은 재작성 가능으로 판단하지 마세요.
6. 재작성 불가능한 경우 재작성 질문을 만들지 말고 필요한 정보만 안내하세요.
7. 반드시 한국어로만 답변하세요.
8. 영어를 섞지 마세요.
9. 출력 형식은 반드시 지키세요.

출력 형식:
판단: 재작성 가능 / 추가 확인 필요
재작성 질문:
추가로 필요한 정보:

[이전 질문]
{previous_question}

[추가 답변]
{additional_answer}
"""

In [ ]:
improved_qa_test_questions = [
    # 기존 QA 질문
    {
        "case_id": "IQA-01",
        "source_type": "existing",
        "category": "금액 단위 보존",
        "context": """
사업예산: 11,270,000,000원
본 사업의 예산은 부가가치세를 포함한 금액입니다.
""",
        "question": "이 사업의 예산은 얼마인가요?",
        "check_points": ["11,270,000,000원", "억/조 임의 환산 금지"]
    },
    {
        "case_id": "IQA-02",
        "source_type": "existing",
        "category": "항목 누락 방지",
        "context": """
구축 대상 시스템은 다음과 같습니다.
1. 포털 시스템
2. 학사 시스템
3. 수강신청 시스템
4. 전자결재 시스템
5. 규정관리 시스템
6. 검색 시스템
7. 챗봇 시스템
8. 모바일 시스템
9. 통계 시스템
""",
        "question": "구축 대상 시스템 목록을 모두 알려주세요.",
        "check_points": ["9개 항목 모두 포함", "항목명 왜곡 없음"]
    },
    {
        "case_id": "IQA-03",
        "source_type": "existing",
        "category": "가격점수 가드레일",
        "context": """
가격평가는 입찰가격 평점산식에 따라 산정합니다.
다만 현재 문서에는 예정가격, 입찰가격, 평점산식의 구체적인 입력값이 모두 제시되어 있지 않습니다.
""",
        "question": "가격점수를 계산해주세요.",
        "check_points": ["계산하지 않음", "입찰가격", "예정가격", "평점산식"]
    },

    # 신규 QA 질문
    {
        "case_id": "IQA-04",
        "source_type": "new",
        "category": "가격점수 산식 일부 부족",
        "context": """
가격평가 배점은 20점입니다.
입찰가격은 95,000,000원입니다.
다만 예정가격과 가격점수 평점산식은 문서에 제시되어 있지 않습니다.
""",
        "question": "가격점수를 계산해 주세요.",
        "check_points": ["계산하지 않음", "예정가격", "평점산식"]
    },
    {
        "case_id": "IQA-05",
        "source_type": "new",
        "category": "기간 원문 보존",
        "context": """
사업기간: 계약체결일로부터 240일 이내
단, 발주기관의 사정에 따라 일정은 조정될 수 있습니다.
""",
        "question": "사업기간은 어떻게 되나요?",
        "check_points": ["계약체결일로부터 240일 이내", "임의 날짜 변환 금지"]
    },
    {
        "case_id": "IQA-06",
        "source_type": "new",
        "category": "문서 근거 부족",
        "context": """
본 사업은 정보시스템 고도화를 목표로 하며, 세부 기능은 제안요청서의 과업내용서를 따릅니다.
""",
        "question": "유지보수 인력은 몇 명 투입해야 하나요?",
        "check_points": ["문서 근거 부족", "추측하지 않음"]
    }
]

improved_qa_test_df = pd.DataFrame(improved_qa_test_questions)
improved_qa_test_df

In [ ]:
improved_user_facing_test_questions = [
    # 기존 사용자 응답형 질문
    {
        "case_id": "IUF-01",
        "source_type": "existing",
        "category": "입찰 적합도 안내",
        "question": "우리 회사가 이 입찰에 적합한지 봐줘.",
        "check_points": ["회사 정보", "유사 사업 수행 실적", "보유 인력", "기술 역량", "내부 라벨 미노출"]
    },
    {
        "case_id": "IUF-02",
        "source_type": "existing",
        "category": "제안서 작성 안내",
        "question": "제안서에 쓸 만하게 정리해줘.",
        "check_points": ["대상 RFP", "회사 소개", "회사 강점", "유사 사업 실적", "수행 전략"]
    },
    {
        "case_id": "IUF-03",
        "source_type": "existing",
        "category": "모호한 요약 요청 안내",
        "question": "이거 핵심만 뽑아줘.",
        "check_points": ["대상 문서", "요약 범위", "예시 질문"]
    },
    {
        "case_id": "IUF-04",
        "source_type": "existing",
        "category": "가격점수 계산 안내",
        "question": "가격점수 계산해줘.",
        "check_points": ["입찰가격", "예정가격", "평점산식", "평가 배점", "임의 계산 금지"]
    },

    # 신규 사용자 응답형 질문
    {
        "case_id": "IUF-05",
        "source_type": "new",
        "category": "비교 대상 부족 안내",
        "question": "이 사업이랑 비슷한 사업 비교해줘.",
        "check_points": ["비교 대상 문서", "비교 기준", "재질문"]
    },
    {
        "case_id": "IUF-06",
        "source_type": "new",
        "category": "회사 정보 부족 안내",
        "question": "우리 회사 강점 기준으로 제안 전략 써줘.",
        "check_points": ["회사 강점", "유사 실적", "수행 전략", "대상 RFP"]
    },
    {
        "case_id": "IUF-07",
        "source_type": "new",
        "category": "요약 범위 부족 안내",
        "question": "중요한 부분만 보고서 형식으로 정리해줘.",
        "check_points": ["대상 문서", "요약 범위", "보고서 형식"]
    }
]

improved_user_facing_test_df = pd.DataFrame(improved_user_facing_test_questions)
improved_user_facing_test_df

In [ ]:
improved_rewrite_test_cases = [
    # 기존 재작성 질문
    {
        "case_id": "IRW-01",
        "source_type": "existing",
        "category": "비교 질문 재작성 가능",
        "previous_question": "이 사업과 비교해주세요.",
        "additional_answer": "경희대 산학협력단 정보시스템 운영 용역과 비교해주세요.",
        "check_points": ["재작성 가능", "비교 대상 포함"]
    },
    {
        "case_id": "IRW-02",
        "source_type": "existing",
        "category": "입찰 적합도 추가 정보 부족",
        "previous_question": "우리 회사가 이 입찰에 적합한지 봐줘.",
        "additional_answer": "그냥 알아서 해줘.",
        "check_points": ["추가 확인 필요", "회사 정보 필요", "재작성 질문 생성 금지"]
    },
    {
        "case_id": "IRW-03",
        "source_type": "existing",
        "category": "수행 전략 재작성 가능",
        "previous_question": "수행 전략 작성해줘.",
        "additional_answer": "차세대 포털·학사 정보시스템 구축사업 기준으로 작성해줘.",
        "check_points": ["재작성 가능", "사업명 포함"]
    },
    {
        "case_id": "IRW-04",
        "source_type": "existing",
        "category": "비교 대상 부족",
        "previous_question": "이 사업이랑 다른 사업 비교해줘.",
        "additional_answer": "그냥 비슷한 걸로 해줘.",
        "check_points": ["추가 확인 필요", "비교 대상 필요", "재작성 질문 생성 금지"]
    },

    # 신규 재작성 질문
    {
        "case_id": "IRW-05",
        "source_type": "new",
        "category": "가격점수 재작성 정보부족",
        "previous_question": "가격점수 계산해줘.",
        "additional_answer": "입찰가격은 있는데 산식은 몰라.",
        "check_points": ["추가 확인 필요", "예정가격", "평점산식", "재작성 질문 생성 금지"]
    },
    {
        "case_id": "IRW-06",
        "source_type": "new",
        "category": "제안서 작성 재작성 가능",
        "previous_question": "제안서 초안 작성해줘.",
        "additional_answer": "고려대학교 차세대 포털·학사 정보시스템 구축사업의 사업 목적과 주요 요구사항 중심으로 작성해줘.",
        "check_points": ["재작성 가능", "사업명", "사업 목적", "주요 요구사항"]
    },
    {
        "case_id": "IRW-07",
        "source_type": "new",
        "category": "회사 적합도 재작성 가능",
        "previous_question": "우리 회사가 이 사업에 맞는지 봐줘.",
        "additional_answer": "우리 회사는 공공기관 정보시스템 구축 경험이 있고, Java/Spring 개발자 5명과 PM 1명을 투입할 수 있어.",
        "check_points": ["재작성 가능", "공공기관 정보시스템 구축 경험", "Java/Spring", "투입 인력"]
    }
]

improved_rewrite_test_df = pd.DataFrame(improved_rewrite_test_cases)
improved_rewrite_test_df

In [ ]:
improved_qa_results = []

for model in final_candidate_models:
    print(f"개선 QA 테스트 실행 중: {model}")
    
    for item in improved_qa_test_questions:
        prompt = improved_rag_qa_prompt.format(
            context=item["context"],
            question=item["question"]
        )
        
        result = ask_ollama(
            prompt=prompt,
            model=model,
            temperature=0.0,
            timeout=120
        )
        
        improved_qa_results.append({
            "model": model,
            "test_type": "Improved QA",
            "case_id": item["case_id"],
            "source_type": item["source_type"],
            "category": item["category"],
            "question": item["question"],
            "check_points": ", ".join(item["check_points"]),
            "success": result["success"],
            "elapsed": round(result["elapsed"], 3),
            "answer": result["answer"],
            "error": result["error"]
        })

improved_qa_results_df = pd.DataFrame(improved_qa_results)
improved_qa_results_df

In [ ]:
improved_user_facing_results = []

for model in final_candidate_models:
    print(f"개선 사용자 응답형 안내 테스트 실행 중: {model}")
    
    for item in improved_user_facing_test_questions:
        prompt = improved_user_facing_clarification_prompt.format(
            question=item["question"]
        )
        
        result = ask_ollama(
            prompt=prompt,
            model=model,
            temperature=0.0,
            timeout=120
        )
        
        improved_user_facing_results.append({
            "model": model,
            "test_type": "Improved User-Facing Clarification",
            "case_id": item["case_id"],
            "source_type": item["source_type"],
            "category": item["category"],
            "question": item["question"],
            "check_points": ", ".join(item["check_points"]),
            "success": result["success"],
            "elapsed": round(result["elapsed"], 3),
            "answer": result["answer"],
            "error": result["error"]
        })

improved_user_facing_results_df = pd.DataFrame(improved_user_facing_results)
improved_user_facing_results_df

In [ ]:
improved_rewrite_results = []

for model in final_candidate_models:
    print(f"개선 질문 재작성 테스트 실행 중: {model}")
    
    for item in improved_rewrite_test_cases:
        prompt = improved_rewrite_prompt.format(
            previous_question=item["previous_question"],
            additional_answer=item["additional_answer"]
        )
        
        result = ask_ollama(
            prompt=prompt,
            model=model,
            temperature=0.0,
            timeout=120
        )
        
        improved_rewrite_results.append({
            "model": model,
            "test_type": "Improved Rewrite",
            "case_id": item["case_id"],
            "source_type": item["source_type"],
            "category": item["category"],
            "previous_question": item["previous_question"],
            "additional_answer": item["additional_answer"],
            "check_points": ", ".join(item["check_points"]),
            "success": result["success"],
            "elapsed": round(result["elapsed"], 3),
            "answer": result["answer"],
            "error": result["error"]
        })

improved_rewrite_results_df = pd.DataFrame(improved_rewrite_results)
improved_rewrite_results_df

In [ ]:
improved_all_results_df = pd.concat(
    [
        improved_qa_results_df,
        improved_user_facing_results_df,
        improved_rewrite_results_df
    ],
    ignore_index=True,
    sort=False
)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

improved_all_results_df[
    [
        "model",
        "test_type",
        "case_id",
        "source_type",
        "category",
        "success",
        "elapsed",
        "answer"
    ]
]

In [ ]:
improved_time_summary_df = improved_all_results_df.groupby("model")["elapsed"].agg(
    count="count",
    mean="mean",
    min="min",
    max="max"
).reset_index()

improved_time_summary_df = improved_time_summary_df.sort_values("mean")
improved_time_summary_df

In [ ]:
improved_time_by_type_df = improved_all_results_df.groupby(
    ["model", "test_type"]
)["elapsed"].agg(
    count="count",
    mean="mean",
    min="min",
    max="max"
).reset_index()

improved_time_by_type_df.sort_values(["test_type", "mean"])

In [ ]:
improved_manual_eval_df = improved_all_results_df[
    [
        "model",
        "test_type",
        "case_id",
        "source_type",
        "category",
        "answer"
    ]
].copy()

improved_manual_eval_df["manual_result"] = ""
improved_manual_eval_df["memo"] = ""

improved_manual_eval_df

In [ ]:
improved_eval_map = {
    # =========================
    # Improved QA - gemma3:4b
    # =========================
    ("gemma3:4b", "IQA-01"): ("실패", "부가가치세 포함 금액인데 포함되지 않은 금액이라고 반대로 답변함"),
    ("gemma3:4b", "IQA-02"): ("통과", "9개 구축 대상 시스템을 모두 누락 없이 제시함"),
    ("gemma3:4b", "IQA-03"): ("통과", "가격점수 계산을 수행하지 않고 예정가격, 입찰가격, 평점산식 부족을 명확히 언급함"),
    ("gemma3:4b", "IQA-04"): ("부분 통과", "계산하지 않은 점은 적절하나 문서에 있는 가격 평가 배점까지 없다고 잘못 언급함"),
    ("gemma3:4b", "IQA-05"): ("통과", "사업기간을 원문 의미에 맞게 제시함"),
    ("gemma3:4b", "IQA-06"): ("부분 통과", "추측하지 않은 점은 적절하나 문서 근거 부족을 직접적으로 설명하지 않고 과업내용서 확인으로만 안내함"),

    # =========================
    # Improved QA - qwen2.5:3b
    # =========================
    ("qwen2.5:3b", "IQA-01"): ("부분 통과", "원문 금액은 포함했으나 한글 금액 병기를 추가했고 변환값도 부정확함"),
    ("qwen2.5:3b", "IQA-02"): ("통과", "9개 구축 대상 시스템을 모두 누락 없이 제시함"),
    ("qwen2.5:3b", "IQA-03"): ("부분 통과", "부족 정보는 잘 짚었으나 [검색된 문서 내용] 같은 내부 표현을 노출함"),
    ("qwen2.5:3b", "IQA-04"): ("통과", "예정가격과 가격점수 평점산식 부족으로 계산 불가를 적절히 안내함"),
    ("qwen2.5:3b", "IQA-05"): ("실패", "계약체결일로부터 240일 이내가 명시되어 있는데 특정 기간이 없다고 잘못 답변함"),
    ("qwen2.5:3b", "IQA-06"): ("통과", "문서에 유지보수 인력 정보가 없음을 명확히 설명하고 추측하지 않음"),

    # =========================
    # Improved QA - exaone3.5:2.4b
    # =========================
    ("exaone3.5:2.4b", "IQA-01"): ("통과", "예산과 부가가치세 포함 여부를 정확히 답변함"),
    ("exaone3.5:2.4b", "IQA-02"): ("통과", "9개 구축 대상 시스템을 모두 누락 없이 제시함"),
    ("exaone3.5:2.4b", "IQA-03"): ("통과", "가격점수 계산에 필요한 입찰가격, 예정가격, 평점산식, 배점을 모두 요구함"),
    ("exaone3.5:2.4b", "IQA-04"): ("부분 통과", "최종적으로 계산 불가 판단은 맞지만 계산해 드리겠다는 시작 문장이 혼동을 줄 수 있음"),
    ("exaone3.5:2.4b", "IQA-05"): ("통과", "사업기간을 문서 근거에 맞게 답변함"),
    ("exaone3.5:2.4b", "IQA-06"): ("통과", "문서에 유지보수 인력 정보가 없음을 명확히 설명하고 추측하지 않음"),

    # =========================
    # Improved User-Facing - gemma3:4b
    # =========================
    ("gemma3:4b", "IUF-01"): ("통과", "입찰 적합도 판단에 필요한 회사 정보, 유사 실적, 보유 인력, 기술 역량을 요구함"),
    ("gemma3:4b", "IUF-02"): ("통과", "제안서 작성에 필요한 대상 RFP, 회사 소개, 강점, 실적, 수행 전략을 요구함"),
    ("gemma3:4b", "IUF-03"): ("통과", "모호한 요약 요청에 대해 대상 문서와 요약 범위를 요구함"),
    ("gemma3:4b", "IUF-04"): ("부분 통과", "가격점수 계산에 필요한 정보는 요구했으나 예시에서 임의 평가 구조를 제시함"),
    ("gemma3:4b", "IUF-05"): ("부분 통과", "비교 기준을 요구한 점은 좋지만 비교 대상 문서보다 귀사 사업 중심으로 흐름"),
    ("gemma3:4b", "IUF-06"): ("부분 통과", "회사 강점과 유사 실적은 요구했으나 대상 RFP와 수행 전략 정보 요구가 다소 약함"),
    ("gemma3:4b", "IUF-07"): ("부분 통과", "대상 문서와 보고서 주요 항목을 요구했으나 예시 질문 제시는 부족함"),

    # =========================
    # Improved User-Facing - qwen2.5:3b
    # =========================
    ("qwen2.5:3b", "IUF-01"): ("부분 통과", "필요 정보는 일부 짚었으나 판정 필요합니다라는 내부 라벨성 표현이 남음"),
    ("qwen2.5:3b", "IUF-02"): ("부분 통과", "대상 RFP와 회사 실적을 요구했으나 답변이 짧고 수행 전략 요구가 약함"),
    ("qwen2.5:3b", "IUF-03"): ("실패", "모호한 요약 요청인데 회사 주요 사업 분야를 요구하여 질문 유형을 잘못 판단함"),
    ("qwen2.5:3b", "IUF-04"): ("부분 통과", "입찰가격과 예정가격은 요구했으나 판정 필요합니다 표현이 있고 평점산식 요구가 빠짐"),
    ("qwen2.5:3b", "IUF-05"): ("부분 통과", "비교에 필요한 추가 정보를 요구했으나 비교 대상 문서와 비교 기준 요구가 구체적이지 않음"),
    ("qwen2.5:3b", "IUF-06"): ("실패", "정보부족 안내가 아니라 제안 전략 작성 방향으로 답변을 진행함"),
    ("qwen2.5:3b", "IUF-07"): ("실패", "보고서 요약 요청인데 회사 사업 분야와 입찰 자격을 물어 질문 유형을 잘못 판단함"),

    # =========================
    # Improved User-Facing - exaone3.5:2.4b
    # =========================
    ("exaone3.5:2.4b", "IUF-01"): ("통과", "입찰 적합도 판단에 필요한 회사 정보, 유사 실적, 인력, 기술 역량을 요구함"),
    ("exaone3.5:2.4b", "IUF-02"): ("부분 통과", "제안서 작성 요소를 잘 설명했지만 정보부족 안내보다 일반 작성 가이드로 흐름"),
    ("exaone3.5:2.4b", "IUF-03"): ("실패", "모호한 요약 요청인데 RFP 영어 설명과 회사 기술 역량 요구로 흐름"),
    ("exaone3.5:2.4b", "IUF-04"): ("부분 통과", "가격점수 계산에 필요한 정보를 요구했지만 평점산식 표현이 직접적이지 않고 답변이 장황함"),
    ("exaone3.5:2.4b", "IUF-05"): ("부분 통과", "비교 대상 사업과 세부 사항을 요구했으나 예시가 회사 프로젝트 사례 중심으로 흐름"),
    ("exaone3.5:2.4b", "IUF-06"): ("부분 통과", "회사 강점과 유사 프로젝트 경험을 요구했으나 대상 RFP 정보 요구가 약함"),
    ("exaone3.5:2.4b", "IUF-07"): ("부분 통과", "대상 문서와 핵심 요소를 요구했으나 비교 대상 문서까지 요구해 질문 의도와 다소 어긋남"),

    # =========================
    # Improved Rewrite - gemma3:4b
    # =========================
    ("gemma3:4b", "IRW-01"): ("부분 통과", "비교 대상은 포함했으나 귀사 경쟁력 평가처럼 원 질문보다 범위가 확장됨"),
    ("gemma3:4b", "IRW-02"): ("실패", "그냥 알아서 해줘는 정보부족인데 재작성 가능으로 오판함"),
    ("gemma3:4b", "IRW-03"): ("통과", "사업명을 포함하여 수행 전략 질문으로 재작성함"),
    ("gemma3:4b", "IRW-04"): ("실패", "비교 대상이 부족한데 A 사업과 B 사업이라는 임의 재작성 질문을 생성함"),
    ("gemma3:4b", "IRW-05"): ("실패", "산식이 없다고 했는데 재작성 가능으로 오판하고 재작성 질문을 생성함"),
    ("gemma3:4b", "IRW-06"): ("통과", "고려대학교 사업명, 사업 목적, 주요 요구사항 중심의 제안서 초안 질문으로 재작성함"),
    ("gemma3:4b", "IRW-07"): ("통과", "공공기관 정보시스템 구축 경험과 Java/Spring 인력 정보를 포함하여 적합도 질문으로 재작성함"),

    # =========================
    # Improved Rewrite - qwen2.5:3b
    # =========================
    ("qwen2.5:3b", "IRW-01"): ("부분 통과", "비교 대상은 포함했으나 판단이 재작성 가능과 추가 확인 필요로 혼재됨"),
    ("qwen2.5:3b", "IRW-02"): ("부분 통과", "재작성 질문을 만들지 않은 점은 좋지만 판단 형식이 빠지고 필요한 정보가 다소 일반적임"),
    ("qwen2.5:3b", "IRW-03"): ("부분 통과", "재작성 질문은 만들었지만 판단 혼재와 내부 입력 구조 노출 문제가 있음"),
    ("qwen2.5:3b", "IRW-04"): ("통과", "비교 대상 부족을 인식하고 추가 정보를 요구함"),
    ("qwen2.5:3b", "IRW-05"): ("부분 통과", "추가 확인 필요 판단은 맞지만 재작성 질문을 생성했고 예정가격 요구가 빠짐"),
    ("qwen2.5:3b", "IRW-06"): ("실패", "재작성 가능 케이스인데 추가 확인 필요로 판단함"),
    ("qwen2.5:3b", "IRW-07"): ("부분 통과", "재작성 가능 판단은 맞지만 외국어 문자 혼입과 추가 정보 요구가 남음"),

    # =========================
    # Improved Rewrite - exaone3.5:2.4b
    # =========================
    ("exaone3.5:2.4b", "IRW-01"): ("부분 통과", "재작성 가능 판단은 맞지만 비교 대상인 경희대 사업명이 명확히 반영되지 않음"),
    ("exaone3.5:2.4b", "IRW-02"): ("통과", "정보부족 케이스를 재작성 불가능으로 판단하고 추가 확인을 요구함"),
    ("exaone3.5:2.4b", "IRW-03"): ("통과", "사업 기준 수행 전략 질문으로 재작성함"),
    ("exaone3.5:2.4b", "IRW-04"): ("통과", "비교 대상 부족을 재작성 불가능으로 판단하고 필요한 정보를 요구함"),
    ("exaone3.5:2.4b", "IRW-05"): ("부분 통과", "재작성 불가능 판단은 맞지만 가격점수 계산에 필요한 정보가 RFP 가격평가 맥락과 다소 어긋남"),
    ("exaone3.5:2.4b", "IRW-06"): ("부분 통과", "재작성 가능 판단은 맞지만 RFP 영어 설명과 과도한 항목 확장이 있음"),
    ("exaone3.5:2.4b", "IRW-07"): ("부분 통과", "재작성 가능 판단은 맞지만 RFP 영어 설명이 포함되고 추가 정보 요구가 다소 장황함"),
}

def apply_improved_manual_eval(row):
    key = (row["model"], row["case_id"])
    return pd.Series(improved_eval_map.get(key, ("", "")))

improved_manual_eval_df[["manual_result", "memo"]] = improved_manual_eval_df.apply(
    apply_improved_manual_eval,
    axis=1
)

improved_manual_eval_df

In [ ]:
score_map = {
    "통과": 1.0,
    "부분 통과": 0.5,
    "실패": 0.0
}

improved_manual_eval_df["score"] = improved_manual_eval_df["manual_result"].map(score_map)

improved_manual_eval_df[
    [
        "model",
        "test_type",
        "case_id",
        "source_type",
        "category",
        "manual_result",
        "score",
        "memo"
    ]
]

In [ ]:
improved_type_score_summary_df = improved_manual_eval_df.groupby(
    ["model", "test_type"]
).agg(
    total_cases=("case_id", "count"),
    pass_count=("manual_result", lambda x: (x == "통과").sum()),
    partial_count=("manual_result", lambda x: (x == "부분 통과").sum()),
    fail_count=("manual_result", lambda x: (x == "실패").sum()),
    total_score=("score", "sum"),
    avg_score=("score", "mean")
).reset_index()

improved_type_score_summary_df.sort_values(["test_type", "avg_score"], ascending=[True, False])

In [ ]:
improved_overall_score_summary_df = improved_manual_eval_df.groupby("model").agg(
    total_cases=("case_id", "count"),
    pass_count=("manual_result", lambda x: (x == "통과").sum()),
    partial_count=("manual_result", lambda x: (x == "부분 통과").sum()),
    fail_count=("manual_result", lambda x: (x == "실패").sum()),
    total_score=("score", "sum"),
    avg_score=("score", "mean")
).reset_index()

improved_overall_score_summary_df = improved_overall_score_summary_df.sort_values(
    "avg_score",
    ascending=False
)

improved_overall_score_summary_df

8. 최종 결론

### 기본 테스트 수동평가 (평균 점수)

| 모델 | 테스트 수 | 평균 점수 | 판정 |
|---|---|---|---|
| gemma3:4b | 11 | 0.727 | 품질 중심 주력 후보 |
| qwen2.5:3b | 11 | 0.682 | 속도 중심 대안 후보 |
| exaone3.5:2.4b | 11 | 0.636 | 보조 후보 |
| llama3.2:3b | 11 | 0.273 | 제외 |
| phi3.5 | 11 | 0.273 | 제외 |

### 사용자 응답형 안내 테스트 (평균 점수)

| 모델 | 평균 점수 |
|---|---|
| exaone3.5:2.4b | 0.625 |
| gemma3:4b | 0.625 |
| qwen2.5:3b | 0.250 |
| llama3.2:3b | 0.000 |
| phi3.5 | 0.000 |

### 개선 프롬프트 종합 평가 (최종 3종)

| 모델 | 테스트 수 | 통과 | 부분 통과 | 실패 | 평균 점수 |
|---|---|---|---|---|---|
| exaone3.5:2.4b | 20 | 9 | 10 | 1 | 0.700 |
| gemma3:4b | 20 | 9 | 7 | 4 | 0.625 |
| qwen2.5:3b | 20 | 4 | 11 | 5 | 0.475 |

### 응답 속도 비교

| 모델 | 평균 응답시간 |
|---|---|
| qwen2.5:3b | 3.11초 |
| exaone3.5:2.4b | 3.28초 |
| gemma3:4b | 4.19초 |

### 후보 정리 — RAG 연동 전 Generation 단독 실험 기준

| 모델 | 역할 | 강점 | 약점 |
|---|---|---|---|
| gemma3:4b | 품질 중심 주력 후보 | QA 안정성, 자연스러운 한국어 | 응답 속도 느림 |
| qwen2.5:3b | 속도 중심 대안 후보 | 빠른 응답, QA 안정성 | 사용자 안내 품질 낮음 |
| exaone3.5:2.4b | 보조 후보 | 사용자 안내 생성 능력 | 질의 판단 안정성 보완 필요 |
| llama3.2:3b | 제외 | - | 전반적 품질 미달 |
| phi3.5 | 제외 | - | 전반적 품질 미달 |

> 이 노트북은 RAG 연동 전 Generation 단독 실험입니다. RAG 연동 후 상위 모델 비교는 `upper_model_comparison.ipynb`에서 진행합니다.
